# Day 5 — Turning the Cycle Into a Reusable Function
**Mahesha Gonal — Refrigeration Simulation Portfolio**

**In plain terms:** Days 2–4 calculated one cycle by typing out the same steps each time. Day 5 wraps those steps into a single reusable function — give it an evaporator temperature and a condenser temperature, and it hands back COP, mass flow rate, and all four state points. Every notebook from here on reuses this same function instead of repeating the maths.

This day also introduces **mass flow rate** — the original .py file printed `m_dot` but never explained it: it's *how many kilograms of refrigerant per second must circulate* to deliver a fixed amount of cooling (100 W, as a reference duty here). This number is what compressor and capillary-tube sizing are actually built around.

**Refrigerant:** R600a (isobutane — the IFB/LG charge) | T_evap = -25°C | T_cond = 40°C

In [1]:
# CELL 1 — Install dependencies (run this first, every time)
!pip install coolprop matplotlib numpy PyGithub -q

In [2]:
# CELL 2 — Imports
import CoolProp.CoolProp as CP

In [3]:
# CELL 3 — Reusable cycle function
def calculate_cycle(T_evap_C, T_cond_C, refrigerant="R600a"):
    T_evap = T_evap_C + 273.15
    T_cond = T_cond_C + 273.15
    P_evap = CP.PropsSI("P","T",T_evap,"Q",1,refrigerant)
    P_cond = CP.PropsSI("P","T",T_cond,"Q",1,refrigerant)
    h1 = CP.PropsSI("H","T",T_evap,"Q",1,refrigerant)
    s1 = CP.PropsSI("S","T",T_evap,"Q",1,refrigerant)
    h2 = CP.PropsSI("H","P",P_cond,"S",s1,refrigerant)
    h3 = CP.PropsSI("H","T",T_cond,"Q",0,refrigerant)
    h4 = h3
    COP   = (h1-h4)/(h2-h1)
    m_dot = 100/(h1-h4)   # kg/s of refrigerant needed to deliver 100 W of cooling
    return COP, m_dot, h1, h2, h3, h4, P_evap, P_cond

In [4]:
# CELL 4 — Run it for R600a
COP, m_dot, h1, h2, h3, h4, P_evap, P_cond = calculate_cycle(-25, 40)
print(f"COP:    {COP:.3f}")
print(f"m_dot:  {m_dot*1000:.4f} g/s   (refrigerant flow needed for 100 W cooling)")
print(f"P_evap: {P_evap/1e5:.2f} bar    P_cond: {P_cond/1e5:.2f} bar")

COP:    2.749
m_dot:  0.4450 g/s   (refrigerant flow needed for 100 W cooling)
P_evap: 0.58 bar    P_cond: 5.31 bar


**What this number means:** a flow rate of a few hundredths of a gram per second sounds tiny, but household refrigerator compressors really do move that little mass — this is exactly why total refrigerant charge in an R600a fridge is only 40-90 grams, not kilograms. Getting this number right is what lets a designer size the compressor's displacement and pick the right capillary tube bore in the first place.

In [ ]:
# FINAL CELL — Push this notebook to GitHub
from github import Github, Auth
from google.colab import userdata, _message
import json

token = userdata.get('GITHUB_TOKEN')
auth = Auth.Token(token)
g = Github(auth=auth)
repo = g.get_repo("MaheshaGonal/refrigeration-simulation-python")

nb_data = _message.blocking_request('get_ipynb', request='', timeout_sec=30)
content = json.dumps(nb_data['ipynb'], indent=1)

filename = "day05_cop_function.ipynb"

try:
    existing = repo.get_contents(filename)
    repo.update_file(filename, "Add Day 05 - reusable COP function + mass flow rate", content, existing.sha)
    print("Updated existing file on GitHub")
except Exception:
    repo.create_file(filename, "Add Day 05 - reusable COP function + mass flow rate", content)
    print("Created new file on GitHub")

print("GitHub repo: https://github.com/MaheshaGonal/refrigeration-simulation-python")
